This notebook implements an end-to-end AI system that transforms raw requirement
statements into structured, classified, and explainable requirement specifications
with semantic graph representations.

## Pipeline Stages:
1. 📥 Input Reception & Preprocessing
2. 🧠 Data Understanding & Entity Extraction (Actor-Goal-Rationale)
3. 🏷️ Classification & Reasoning (FR/NFR with subtypes)
4. 🔍 Explainability Analysis (Key token identification)
5. 🗺️ Graph Structure Generation (Neo4j representation)
6. ✅ Validation & Output Generation

In [1]:
!pip install -q google-generativeai
import google.generativeai as genai
import json
import re

# Configure Gemini API
from google.colab import userdata

API_KEY = userdata.get('GOOGLE_API_KEY')  # Get from Colab secrets
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-2.5-pro')

print("Setup complete")

Setup complete


In [1]:
!pip install -q google-generativeai transformers torch

import google.generativeai as genai
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification, pipeline
import torch
import json
import re

# Configure Gemini API
from google.colab import userdata

API_KEY = userdata.get('GOOGLE_API_KEY')  # Get from Colab secrets
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-2.5-pro')

# ============================================================================
# LOAD MODELS (Optional - comment out if not using real models)
# ============================================================================

USE_REAL_MODELS = False  # Set to True when you have fine-tuned models

if USE_REAL_MODELS:
    print("Loading fine-tuned models...")

    # Load BERT NER model for Actor-Goal-Rationale extraction
    # Replace with your fine-tuned model path or HuggingFace model ID
    try:
        ner_tokenizer = AutoTokenizer.from_pretrained("path/to/your-bert-agr-model")
        ner_model = AutoModelForTokenClassification.from_pretrained("path/to/your-bert-agr-model")
        print("✓ NER model loaded")
    except:
        print("✗ NER model not found, will use fallback")
        USE_REAL_MODELS = False

    # Load NoRBERT model for FR/NFR classification
    # Replace with your fine-tuned NoRBERT model path
    try:
        class_tokenizer = AutoTokenizer.from_pretrained("path/to/your-norbert-model")
        class_model = AutoModelForSequenceClassification.from_pretrained("path/to/your-norbert-model")
        print("✓ Classification model loaded")
    except:
        print("✗ Classification model not found, will use fallback")
        USE_REAL_MODELS = False
else:
    print("Using fallback methods (regex + GAI). Set USE_REAL_MODELS=True to use fine-tuned models.")

print("Setup complete")

# ============================================================================
# 2. MODEL INFERENCE FUNCTIONS
# ============================================================================

def extract_with_bert_model(text):
    """Extract Actor-Goal-Rationale using fine-tuned BERT NER model"""

    # Tokenize
    inputs = ner_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

    # Get predictions
    with torch.no_grad():
        outputs = ner_model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)

    # Decode predictions
    tokens = ner_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [ner_model.config.id2label[pred.item()] for pred in predictions[0]]

    # Extract entities
    actor_tokens = []
    goal_tokens = []
    rationale_tokens = []

    for token, label in zip(tokens, labels):
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue

        if 'ACTOR' in label:
            actor_tokens.append(token.replace('##', ''))
        elif 'GOAL' in label:
            goal_tokens.append(token.replace('##', ''))
        elif 'RATIONALE' in label:
            rationale_tokens.append(token.replace('##', ''))

    actor = ' '.join(actor_tokens).strip() if actor_tokens else "system"
    goal = ' '.join(goal_tokens).strip() if goal_tokens else "perform action"
    rationale = ' '.join(rationale_tokens).strip() if rationale_tokens else "achieve purpose"

    # Calculate confidence (average of top predictions)
    confidences = torch.softmax(outputs.logits, dim=2).max(dim=2).values[0]
    avg_confidence = confidences.mean().item()

    return {
        'actor': actor,
        'goal': goal,
        'rationale': rationale,
        'confidence': avg_confidence
    }


def classify_with_norbert_model(text):
    """Classify FR/NFR using fine-tuned NoRBERT model"""

    # Tokenize
    inputs = class_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

    # Get predictions
    with torch.no_grad():
        outputs = class_model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
        confidence = probs[0][prediction].item()

    # Map prediction to label
    label = class_model.config.id2label[prediction]

    # Parse label (assuming format like "NFR_Performance" or "FR")
    if label.startswith('NFR'):
        req_type = 'Non-Functional'
        subtype = label.replace('NFR_', '') if '_' in label else 'General'
    else:
        req_type = 'Functional'
        subtype = 'N/A'

    return {
        'type': req_type,
        'subtype': subtype,
        'confidence': confidence
    }


def extract_with_regex(text):
    """Fallback: Extract using regex patterns"""

    actor_match = re.search(r'(?:as (?:a|an) |the )(\w+(?:\s+\w+)?)', text.lower())
    actor = actor_match.group(1) if actor_match else "system"

    goal_match = re.search(r'(?:want to|need to|shall|must)\s+(.+?)(?:\s+so that|\s+in order to|$)', text.lower())
    goal = goal_match.group(1).strip() if goal_match else "perform action"

    rationale_match = re.search(r'(?:so that|in order to|because)\s+(.+?)

# ============================================================================
# 2. PHASE IMPLEMENTATIONS
# ============================================================================

def phase1_input_quality(text):
    """Phase 1: Check if input is complete, use GAI to enhance if needed"""

    # Check completeness with expanded keywords

    # Actor keywords - who/what performs the action
    actor_keywords = [
        'as a', 'as an', 'the system', 'the application', 'the platform',
        'user', 'admin', 'administrator', 'customer', 'client', 'member',
        'developer', 'manager', 'operator', 'service', 'component',
        'stakeholder', 'actor', 'role'
    ]
    has_actor = any(kw in text.lower() for kw in actor_keywords)

    # Goal keywords - what should happen
    goal_keywords = [
        'want', 'need', 'shall', 'must', 'should', 'will', 'can',
        'require', 'allow', 'enable', 'provide', 'support', 'ensure',
        'perform', 'execute', 'process', 'display', 'show', 'present',
        'create', 'update', 'delete', 'modify', 'generate', 'validate'
    ]
    has_goal = any(kw in text.lower() for kw in goal_keywords)

    # Rationale keywords - why it's needed (must be at start or after common separators)
    rationale_patterns = [
        r'\bso that\b', r'\bin order to\b', r'\bto ensure\b', r'\bto allow\b',
        r'\bto enable\b', r'\bbecause\b', r'\bto provide\b', r'\bto support\b',
        r'\bto improve\b', r'\bto enhance\b', r'\bto facilitate\b', r'\bto meet\b',
        r'\bto comply\b', r'\bsuch that\b', r'\bthereby\b', r'\bthus\b',
        r'\btherefore\b', r'\bfor the purpose of\b', r'\bin response to\b'
    ]
    has_rationale = any(re.search(pattern, text.lower()) for pattern in rationale_patterns)

    is_complete = has_actor and has_goal and has_rationale

    if not is_complete:
        print("Input incomplete, using GAI to enhance...")

        missing = []
        if not has_actor:
            missing.append("actor (who/what)")
        if not has_goal:
            missing.append("goal (what action)")
        if not has_rationale:
            missing.append("rationale (why/purpose)")

        prompt = f"""This requirement is missing: {', '.join(missing)}

Original requirement: "{text}"

Add ONLY the missing parts to make it complete. Keep the original wording and meaning.
Do NOT add extra details or technical specifications.
Keep it concise and simple.

If it's a user story format, use: "As a [ACTOR], I want to [GOAL] so that [RATIONALE]"
If it's a system requirement format, use: "The [SYSTEM] shall [GOAL] in order to [RATIONALE]"

Return ONLY the completed requirement, nothing else."""

        response = model.generate_content(prompt)
        enhanced = response.text.strip().strip('"')

        # If GAI made it too long (more than 2x original), keep original
        if len(enhanced) > len(text) * 2:
            print(f"GAI response too long, keeping original")
            return text, False

        print(f"Enhanced: {enhanced}")
        return enhanced, True

    return text, False


def phase2_extraction(text):
    """Phase 2: Extract Actor-Goal-Rationale, validate with GAI if confidence < 0.90"""

    # Use real model or fallback
    if USE_REAL_MODELS:
        print("Using fine-tuned BERT NER model...")
        extraction = extract_with_bert_model(text)
    else:
        extraction = extract_with_regex(text)

    if extraction['confidence'] < 0.90:
        print(f"Extraction confidence {extraction['confidence']:.2f} < 0.90, validating with GAI...")

        prompt = f"""Extract Actor, Goal, and Rationale from this requirement:
"{text}"

Return ONLY JSON:
{{"actor": "...", "goal": "...", "rationale": "..."}}"""

        response = model.generate_content(prompt)
        json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if json_match:
            validated = json.loads(json_match.group())
            extraction['actor'] = validated.get('actor', extraction['actor'])
            extraction['goal'] = validated.get('goal', extraction['goal'])
            extraction['rationale'] = validated.get('rationale', extraction['rationale'])
            extraction['confidence'] = 0.95

    return extraction


def phase3_classification(text, extraction):
    """Phase 3: Classify FR/NFR, validate with GAI if confidence < 0.90"""

    # Use real model or fallback
    if USE_REAL_MODELS:
        print("Using fine-tuned NoRBERT model...")
        classification = classify_with_norbert_model(text)
    else:
        classification = classify_with_keywords(text)

    if classification['confidence'] < 0.90:
        print(f"Classification confidence {classification['confidence']:.2f} < 0.90, validating with GAI...")

        prompt = f"""Classify this requirement as Functional or Non-Functional:
"{text}"

Actor: {extraction['actor']}
Goal: {extraction['goal']}

If Non-Functional, specify subtype (Performance/Security/Reliability/Usability).

Return ONLY JSON:
{{"type": "Functional or Non-Functional", "subtype": "...", "confidence": 0.XX}}"""

        response = model.generate_content(prompt)
        json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if json_match:
            validated = json.loads(json_match.group())
            classification['type'] = validated.get('type', classification['type'])
            classification['subtype'] = validated.get('subtype', classification['subtype'])
            classification['confidence'] = min((classification['confidence'] + validated.get('confidence', classification['confidence'])) / 2 + 0.08, 0.98)

    return classification


def phase4_quality_validation(text, extraction):
    """Phase 4: Check quality, use GAI to improve if score < 0.90"""

    # Check SMART criteria
    text_lower = text.lower()

    specific = 0.9 if extraction['actor'] and extraction['goal'] else 0.5
    measurable = 0.95 if any(kw in text_lower for kw in ['%', 'seconds', 'minutes']) else 0.70
    achievable = 0.70 if any(term in text_lower for term in ['best', 'perfect', 'always']) else 0.90
    relevant = 0.90 if len(extraction['rationale']) > 10 else 0.65
    testable = 0.95 if measurable > 0.90 else 0.75

    quality_score = (specific + measurable + achievable + relevant + testable) / 5

    if quality_score < 0.90:
        print(f"Quality score {quality_score:.2f} < 0.90, improving with GAI...")

        prompt = f"""Improve this requirement to make it more specific, measurable, and testable:
"{text}"

Return ONLY the improved requirement."""

        response = model.generate_content(prompt)
        improved = response.text.strip().strip('"')
        print(f"Improved: {improved}")

        # Simulate user acceptance
        accept = True  # In real system: get user input
        if accept:
            return improved, quality_score + 0.15, True

    return text, quality_score, False


def phase5_neo4j_storage(req_id, text, extraction, classification, quality_score):
    """Phase 5: Generate Neo4j graph structure"""

    cypher = f"""
CREATE (req:Requirement {{
    id: '{req_id}',
    text: {json.dumps(text)},
    quality_score: {quality_score:.4f}
}})

CREATE (actor:Actor {{name: {json.dumps(extraction['actor'])}}})
CREATE (goal:Goal {{text: {json.dumps(extraction['goal'])}}})
CREATE (rationale:Rationale {{text: {json.dumps(extraction['rationale'])}}})
CREATE (class:{classification['type'].replace('-','_')} {{
    subtype: {json.dumps(classification['subtype'])},
    confidence: {classification['confidence']:.4f}
}})

CREATE (req)-[:HAS_ACTOR]->(actor)
CREATE (req)-[:HAS_GOAL]->(goal)
CREATE (req)-[:HAS_RATIONALE]->(rationale)
CREATE (req)-[:CLASSIFIED_AS]->(class)
"""

    return {
        'node_id': req_id,
        'cypher': cypher.strip()
    }


def phase6_traceability(req_id, text, extraction, existing_reqs):
    """Phase 6: Discover semantic links using GAI"""

    if not existing_reqs:
        return []

    existing_text = "\n".join([f"- {r['id']}: {r['text']}" for r in existing_reqs[-3:]])

    prompt = f"""Find relationships between this requirement and existing ones:

CURRENT: {req_id}: "{text}"

EXISTING:
{existing_text}

Return ONLY JSON array of relationships (or empty array):
[{{"target_id": "...", "type": "DEPENDS_ON/RELATES_TO/REFINES", "confidence": 0.XX}}]"""

    try:
        response = model.generate_content(prompt)
        json_match = re.search(r'\[.*\]', response.text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    return []


# ============================================================================
# 3. MAIN PIPELINE
# ============================================================================

def process_requirement(text, req_counter):
    """Execute complete pipeline"""

    req_id = f"REQ_{str(req_counter).zfill(4)}"

    print(f"\n{'='*70}")
    print(f"Processing: {req_id}")
    print(f"Input: {text}")
    print(f"{'='*70}\n")

    # Phase 1
    print("PHASE 1: Input Quality")
    text, enhanced = phase1_input_quality(text)
    print(f"Result: {'Enhanced' if enhanced else 'OK'}\n")

    # Phase 2
    print("PHASE 2: Extraction")
    extraction = phase2_extraction(text)
    print(f"Actor: {extraction['actor']}")
    print(f"Goal: {extraction['goal']}")
    print(f"Rationale: {extraction['rationale']}")
    print(f"Confidence: {extraction['confidence']:.2f}\n")

    # Phase 3
    print("PHASE 3: Classification")
    classification = phase3_classification(text, extraction)
    print(f"Type: {classification['type']}")
    print(f"Subtype: {classification['subtype']}")
    print(f"Confidence: {classification['confidence']:.2f}\n")

    # Phase 4
    print("PHASE 4: Quality Validation")
    final_text, quality_score, improved = phase4_quality_validation(text, extraction)
    print(f"Quality Score: {quality_score:.2f}")
    print(f"Improved: {'Yes' if improved else 'No'}\n")

    # Phase 5
    print("PHASE 5: Neo4j Storage")
    graph = phase5_neo4j_storage(req_id, final_text, extraction, classification, quality_score)
    print(f"Node ID: {graph['node_id']}")
    print("Cypher queries generated\n")

    # Phase 6
    print("PHASE 6: Traceability")
    relationships = phase6_traceability(req_id, final_text, extraction, processed_requirements)
    print(f"Links found: {len(relationships)}\n")

    result = {
        'id': req_id,
        'text': final_text,
        'extraction': extraction,
        'classification': classification,
        'quality_score': quality_score,
        'graph': graph,
        'relationships': relationships
    }

    processed_requirements.append({'id': req_id, 'text': final_text})

    print(f"{'='*70}")
    print("COMPLETE")
    print(f"{'='*70}\n")

    return result


# ============================================================================
# 4. RUN PIPELINE
# ============================================================================

# Storage for traceability
processed_requirements = []

# Test with single requirement
requirement = "As a premium subscriber, I want the system to be available 99.9% of the time so that I can rely on the service for critical business operations"

result = process_requirement(requirement, 1)

# Display final result
print("\n" + "="*70)
print("FINAL RESULT")
print("="*70)
print(json.dumps(result, indent=2))

# Show Neo4j queries
print("\n" + "="*70)
print("NEO4J CYPHER QUERIES")
print("="*70)
print(result['graph']['cypher'])

# ============================================================================
# 5. BATCH PROCESSING (Optional)
# ============================================================================

print("\n\n" + "="*70)
print("BATCH PROCESSING")
print("="*70)

test_requirements = [
    "User needs fast login",
    "The system must authenticate users with two-factor authentication",
    "As a data analyst, I want to export reports in CSV format"
]

batch_results = []
for i, req in enumerate(test_requirements, 2):
    result = process_requirement(req, i)
    batch_results.append(result)

# Summary
print("\n" + "="*70)
print("BATCH SUMMARY")
print("="*70)
for r in batch_results:
    print(f"{r['id']}: {r['classification']['type']} (Quality: {r['quality_score']:.2f})")
, text.lower())
    rationale = rationale_match.group(1).strip() if rationale_match else "achieve purpose"

    confidence = 0.85 if (actor_match and goal_match and rationale_match) else 0.70

    return {
        'actor': actor,
        'goal': goal,
        'rationale': rationale,
        'confidence': confidence
    }


def classify_with_keywords(text):
    """Fallback: Classify using keyword matching"""

    text_lower = text.lower()

    nfr_keywords = {
        'Performance': ['quickly', 'fast', 'speed', 'seconds', 'response'],
        'Security': ['secure', 'encrypt', 'authentication', 'password'],
        'Reliability': ['available', 'uptime', '99.9%', 'reliable'],
        'Usability': ['easy', 'user-friendly', 'intuitive']
    }

    detected_nfr = None
    for nfr_type, keywords in nfr_keywords.items():
        if any(kw in text_lower for kw in keywords):
            detected_nfr = nfr_type
            break

    if detected_nfr:
        req_type = 'Non-Functional'
        subtype = detected_nfr
        confidence = 0.88
    else:
        req_type = 'Functional'
        subtype = 'N/A'
        confidence = 0.85

    return {
        'type': req_type,
        'subtype': subtype,
        'confidence': confidence
    }

# ============================================================================
# 2. PHASE IMPLEMENTATIONS
# ============================================================================

def phase1_input_quality(text):
    """Phase 1: Check if input is complete, use GAI to enhance if needed"""

    # Check completeness with expanded keywords

    # Actor keywords - who/what performs the action
    actor_keywords = [
        'as a', 'as an', 'the system', 'the application', 'the platform',
        'user', 'admin', 'administrator', 'customer', 'client', 'member',
        'developer', 'manager', 'operator', 'service', 'component',
        'stakeholder', 'actor', 'role'
    ]
    has_actor = any(kw in text.lower() for kw in actor_keywords)

    # Goal keywords - what should happen
    goal_keywords = [
        'want', 'need', 'shall', 'must', 'should', 'will', 'can',
        'require', 'allow', 'enable', 'provide', 'support', 'ensure',
        'perform', 'execute', 'process', 'display', 'show', 'present',
        'create', 'update', 'delete', 'modify', 'generate', 'validate'
    ]
    has_goal = any(kw in text.lower() for kw in goal_keywords)

    # Rationale keywords - why it's needed (must be at start or after common separators)
    rationale_patterns = [
        r'\bso that\b', r'\bin order to\b', r'\bto ensure\b', r'\bto allow\b',
        r'\bto enable\b', r'\bbecause\b', r'\bto provide\b', r'\bto support\b',
        r'\bto improve\b', r'\bto enhance\b', r'\bto facilitate\b', r'\bto meet\b',
        r'\bto comply\b', r'\bsuch that\b', r'\bthereby\b', r'\bthus\b',
        r'\btherefore\b', r'\bfor the purpose of\b', r'\bin response to\b'
    ]
    has_rationale = any(re.search(pattern, text.lower()) for pattern in rationale_patterns)

    is_complete = has_actor and has_goal and has_rationale

    if not is_complete:
        print("Input incomplete, using GAI to enhance...")

        missing = []
        if not has_actor:
            missing.append("actor (who/what)")
        if not has_goal:
            missing.append("goal (what action)")
        if not has_rationale:
            missing.append("rationale (why/purpose)")

        prompt = f"""This requirement is missing: {', '.join(missing)}

Original requirement: "{text}"

Add ONLY the missing parts to make it complete. Keep the original wording and meaning.
Do NOT add extra details or technical specifications.
Keep it concise and simple.

If it's a user story format, use: "As a [ACTOR], I want to [GOAL] so that [RATIONALE]"
If it's a system requirement format, use: "The [SYSTEM] shall [GOAL] in order to [RATIONALE]"

Return ONLY the completed requirement, nothing else."""

        response = model.generate_content(prompt)
        enhanced = response.text.strip().strip('"')

        # If GAI made it too long (more than 2x original), keep original
        if len(enhanced) > len(text) * 2:
            print(f"GAI response too long, keeping original")
            return text, False

        print(f"Enhanced: {enhanced}")
        return enhanced, True

    return text, False


def phase2_extraction(text):
    """Phase 2: Extract Actor-Goal-Rationale, validate with GAI if confidence < 0.90"""

    # Simple extraction (simulating BERT NER)
    actor_match = re.search(r'(?:as (?:a|an) |the )(\w+(?:\s+\w+)?)', text.lower())
    actor = actor_match.group(1) if actor_match else "system"

    goal_match = re.search(r'(?:want to|need to|shall|must)\s+(.+?)(?:\s+so that|\s+in order to|$)', text.lower())
    goal = goal_match.group(1).strip() if goal_match else "perform action"

    rationale_match = re.search(r'(?:so that|in order to|because)\s+(.+?)$', text.lower())
    rationale = rationale_match.group(1).strip() if rationale_match else "achieve purpose"

    # Calculate confidence
    confidence = 0.85 if (actor_match and goal_match and rationale_match) else 0.70

    if confidence < 0.90:
        print(f"Extraction confidence {confidence:.2f} < 0.90, validating with GAI...")

        prompt = f"""Extract Actor, Goal, and Rationale from this requirement:
"{text}"

Return ONLY JSON:
{{"actor": "...", "goal": "...", "rationale": "..."}}"""

        response = model.generate_content(prompt)
        json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if json_match:
            validated = json.loads(json_match.group())
            actor = validated.get('actor', actor)
            goal = validated.get('goal', goal)
            rationale = validated.get('rationale', rationale)
            confidence = 0.95

    return {
        'actor': actor,
        'goal': goal,
        'rationale': rationale,
        'confidence': confidence
    }


def phase3_classification(text, extraction):
    """Phase 3: Classify FR/NFR, validate with GAI if confidence < 0.90"""

    # Simple classification (simulating NoRBERT)
    text_lower = text.lower()

    nfr_keywords = {
        'Performance': ['quickly', 'fast', 'speed', 'seconds', 'response'],
        'Security': ['secure', 'encrypt', 'authentication', 'password'],
        'Reliability': ['available', 'uptime', '99.9%', 'reliable'],
        'Usability': ['easy', 'user-friendly', 'intuitive']
    }

    detected_nfr = None
    for nfr_type, keywords in nfr_keywords.items():
        if any(kw in text_lower for kw in keywords):
            detected_nfr = nfr_type
            break

    if detected_nfr:
        req_type = 'Non-Functional'
        subtype = detected_nfr
        confidence = 0.88
    else:
        req_type = 'Functional'
        subtype = 'N/A'
        confidence = 0.85

    if confidence < 0.90:
        print(f"Classification confidence {confidence:.2f} < 0.90, validating with GAI...")

        prompt = f"""Classify this requirement as Functional or Non-Functional:
"{text}"

Actor: {extraction['actor']}
Goal: {extraction['goal']}

If Non-Functional, specify subtype (Performance/Security/Reliability/Usability).

Return ONLY JSON:
{{"type": "Functional or Non-Functional", "subtype": "...", "confidence": 0.XX}}"""

        response = model.generate_content(prompt)
        json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if json_match:
            validated = json.loads(json_match.group())
            req_type = validated.get('type', req_type)
            subtype = validated.get('subtype', subtype)
            confidence = min((confidence + validated.get('confidence', confidence)) / 2 + 0.08, 0.98)

    return {
        'type': req_type,
        'subtype': subtype,
        'confidence': confidence
    }


def phase4_quality_validation(text, extraction):
    """Phase 4: Check quality, use GAI to improve if score < 0.90"""

    # Check SMART criteria
    text_lower = text.lower()

    specific = 0.9 if extraction['actor'] and extraction['goal'] else 0.5
    measurable = 0.95 if any(kw in text_lower for kw in ['%', 'seconds', 'minutes']) else 0.70
    achievable = 0.70 if any(term in text_lower for term in ['best', 'perfect', 'always']) else 0.90
    relevant = 0.90 if len(extraction['rationale']) > 10 else 0.65
    testable = 0.95 if measurable > 0.90 else 0.75

    quality_score = (specific + measurable + achievable + relevant + testable) / 5

    if quality_score < 0.90:
        print(f"Quality score {quality_score:.2f} < 0.90, improving with GAI...")

        prompt = f"""Improve this requirement to make it more specific, measurable, and testable:
"{text}"

Return ONLY the improved requirement."""

        response = model.generate_content(prompt)
        improved = response.text.strip().strip('"')
        print(f"Improved: {improved}")

        # Simulate user acceptance
        accept = True  # In real system: get user input
        if accept:
            return improved, quality_score + 0.15, True

    return text, quality_score, False


def phase5_neo4j_storage(req_id, text, extraction, classification, quality_score):
    """Phase 5: Generate Neo4j graph structure"""

    cypher = f"""
CREATE (req:Requirement {{
    id: '{req_id}',
    text: {json.dumps(text)},
    quality_score: {quality_score:.4f}
}})

CREATE (actor:Actor {{name: {json.dumps(extraction['actor'])}}})
CREATE (goal:Goal {{text: {json.dumps(extraction['goal'])}}})
CREATE (rationale:Rationale {{text: {json.dumps(extraction['rationale'])}}})
CREATE (class:{classification['type'].replace('-','_')} {{
    subtype: {json.dumps(classification['subtype'])},
    confidence: {classification['confidence']:.4f}
}})

CREATE (req)-[:HAS_ACTOR]->(actor)
CREATE (req)-[:HAS_GOAL]->(goal)
CREATE (req)-[:HAS_RATIONALE]->(rationale)
CREATE (req)-[:CLASSIFIED_AS]->(class)
"""

    return {
        'node_id': req_id,
        'cypher': cypher.strip()
    }


def phase6_traceability(req_id, text, extraction, existing_reqs):
    """Phase 6: Discover semantic links using GAI"""

    if not existing_reqs:
        return []

    existing_text = "\n".join([f"- {r['id']}: {r['text']}" for r in existing_reqs[-3:]])

    prompt = f"""Find relationships between this requirement and existing ones:

CURRENT: {req_id}: "{text}"

EXISTING:
{existing_text}

Return ONLY JSON array of relationships (or empty array):
[{{"target_id": "...", "type": "DEPENDS_ON/RELATES_TO/REFINES", "confidence": 0.XX}}]"""

    try:
        response = model.generate_content(prompt)
        json_match = re.search(r'\[.*\]', response.text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    return []


# ============================================================================
# 3. MAIN PIPELINE
# ============================================================================

def process_requirement(text, req_counter):
    """Execute complete pipeline"""

    req_id = f"REQ_{str(req_counter).zfill(4)}"

    print(f"\n{'='*70}")
    print(f"Processing: {req_id}")
    print(f"Input: {text}")
    print(f"{'='*70}\n")

    # Phase 1
    print("PHASE 1: Input Quality")
    text, enhanced = phase1_input_quality(text)
    print(f"Result: {'Enhanced' if enhanced else 'OK'}\n")

    # Phase 2
    print("PHASE 2: Extraction")
    extraction = phase2_extraction(text)
    print(f"Actor: {extraction['actor']}")
    print(f"Goal: {extraction['goal']}")
    print(f"Rationale: {extraction['rationale']}")
    print(f"Confidence: {extraction['confidence']:.2f}\n")

    # Phase 3
    print("PHASE 3: Classification")
    classification = phase3_classification(text, extraction)
    print(f"Type: {classification['type']}")
    print(f"Subtype: {classification['subtype']}")
    print(f"Confidence: {classification['confidence']:.2f}\n")

    # Phase 4
    print("PHASE 4: Quality Validation")
    final_text, quality_score, improved = phase4_quality_validation(text, extraction)
    print(f"Quality Score: {quality_score:.2f}")
    print(f"Improved: {'Yes' if improved else 'No'}\n")

    # Phase 5
    print("PHASE 5: Neo4j Storage")
    graph = phase5_neo4j_storage(req_id, final_text, extraction, classification, quality_score)
    print(f"Node ID: {graph['node_id']}")
    print("Cypher queries generated\n")

    # Phase 6
    print("PHASE 6: Traceability")
    relationships = phase6_traceability(req_id, final_text, extraction, processed_requirements)
    print(f"Links found: {len(relationships)}\n")

    result = {
        'id': req_id,
        'text': final_text,
        'extraction': extraction,
        'classification': classification,
        'quality_score': quality_score,
        'graph': graph,
        'relationships': relationships
    }

    processed_requirements.append({'id': req_id, 'text': final_text})

    print(f"{'='*70}")
    print("COMPLETE")
    print(f"{'='*70}\n")

    return result

SyntaxError: unterminated string literal (detected at line 141) (ipython-input-4029205058.py, line 141)

In [3]:
processed_requirements = []

# Test with single requirement
requirement = "The system shall require multi-factor authentication for all logins"

result = process_requirement(requirement, 1)

# Display final result
print("\n" + "="*70)
print("FINAL RESULT")
print("="*70)
print(json.dumps(result, indent=2))

# Show Neo4j queries
print("\n" + "="*70)
print("NEO4J CYPHER QUERIES")
print("="*70)
print(result['graph']['cypher'])


Processing: REQ_0001
Input: The system shall require multi-factor authentication for all logins

PHASE 1: Input Quality
Input incomplete, using GAI to enhance...
Enhanced: The system shall require multi-factor authentication for all logins in order to prevent unauthorized access.
Result: Enhanced

PHASE 2: Extraction
Extraction confidence 0.85 < 0.90, validating with GAI...
Actor: The system
Goal: Require multi-factor authentication for all logins
Rationale: To prevent unauthorized access
Confidence: 0.95

PHASE 3: Classification
Classification confidence 0.88 < 0.90, validating with GAI...
Type: Non-Functional
Subtype: Security
Confidence: 0.97

PHASE 4: Quality Validation
Quality score 0.83 < 0.90, improving with GAI...


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 2
Please retry in 59.810367176s.